In [13]:
from google.colab import files
uploaded = files.upload()

Saving novel.txt to novel.txt


In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### STEP 1: Load Text

In [15]:
# Load text
with open("novel.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

print("Length of text:", len(text))
print(text[:200])

Length of text: 4845
chapter one: the last light before dawn

the rain had not stopped for three days.
it fell in steady curtains across the narrow streets of ashford.
street lamps flickered in the mist like uncertain mem


The text is converted to lowercase to maintain consistency and reduce vocabulary size.

### STEP 2: Create Vocabulary & Sequences

In [16]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

Vocabulary Size: 383


The Tokenizer creates a vocabulary of all unique words in the dataset and assigns each word a unique integer index. The total vocabulary size is calculated and will be used in the embedding and output layers of the model.

In [17]:
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max([len(x) for x in input_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

Training sequences are generated using the n-gram approach. For each sentence, it creates progressively increasing sequences of words. These sequences help the model learn how to predict the next word based on previous words.

Since sequences have different lengths, padding is applied to make them equal in size. Pre-padding adds zeros at the beginning of shorter sequences so that all input data has uniform dimensions for training.

### STEP 3: Create Training Data

In [18]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

The dataset is divided into input features (X) and target labels (y).
X contains all words except the last word in each sequence.
y contains the next word to be predicted.
The target variable is converted into one-hot encoded format for multi-class classification.

### STEP 4A: Build RNN Model

In [19]:
rnn_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len-1),
    SimpleRNN(150),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

The Embedding layer converts word indices into dense vector representations.
The SimpleRNN layer processes sequential data and learns temporal patterns.
The Dense layer with softmax activation outputs probabilities for all words in the vocabulary.
The model is compiled using categorical cross-entropy loss and the Adam optimizer.

### STEP 4B: Build LSTM Model

In [20]:
lstm_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len-1),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

The structure is similar to the RNN model, but it uses an LSTM layer instead of SimpleRNN.
LSTM is capable of learning long-term dependencies using memory cells and gates, making it more effective for sequential text generation.

### STEP 5: Train Models

In [21]:
rnn_model.fit(X, y, epochs=50, verbose=1)

lstm_model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.0381 - loss: 5.9019
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0704 - loss: 5.5416
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.0613 - loss: 5.4565
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0561 - loss: 5.3875
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0812 - loss: 5.1634
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0862 - loss: 5.0870
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0909 - loss: 4.9411
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1115 - loss: 4.7426
Epoch 9/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1605 - loss: 4.5404
Epoch 10/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2075 - loss: 4.2385
Epoch 11/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2077 - loss: 4.0936
Epoch 12/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2915 -

During training, the model performs forward propagation, computes loss, performs backpropagation through time (BPTT), and updates weights using the Adam optimizer.
Training is repeated for 50 epochs.

### STEP 6: Text Generation Function

In [24]:
import random

def generate_text(model, seed_text, next_words, temperature=1.0):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predictions = model.predict(token_list)[0]

        predictions = np.log(predictions + 1e-8) / temperature
        exp_preds = np.exp(predictions)
        predictions = exp_preds / np.sum(exp_preds)

        predicted_index = np.random.choice(len(predictions), p=predictions)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

This function generates new text using the trained model.
It takes a seed sentence as input and predicts the next word iteratively.
At each step, the model outputs probabilities for all words, and the word with the highest probability is selected.
The predicted word is appended to the input sequence to generate continuous text.

### STEP 7: Generate Text

In [26]:
print(generate_text(rnn_model, "the rain had", 15, temperature=0.8))
print(generate_text(lstm_model, "the rain had", 15, temperature=0.8))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
the rain had not stopped for three days truth by thin the weight of unseen eyes sunrise dream
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40m

The generated text demonstrates that both RNN and LSTM models successfully learned sequential patterns from the training dataset. The LSTM model produced more coherent and contextually consistent output due to its ability to handle long-term dependencies. Minor grammatical inconsistencies are due to limited dataset size and probabilistic word prediction.